## **Loan Prediction Machine Learning Model**

In [ ]:
# Import Data Manipulation Libraries
import pandas as pd
import numpy as np
# Import Data Visualization Libraries
import seaborn as sns
import matplotlib.pyplot as plt
# Import Logging
import logging
logging.basicConfig(level = logging.INFO,
                    filename = 'logs/model.log',
                    filemode = 'w',
                    format = '%(name)s - %(levelname)s - %(message)s -%(levelname)s',
                    force = True)

# Import FilterWarning Libraries
import warnings
warnings.filterwarnings(action = 'ignore')
# Import Machine Learning Libraries
from sklearn.preprocessing import MinMaxScaler,RobustScaler,LabelEncoder,OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from scipy.stats.mstats import winsorize

In [ ]:
!pip install "flaml[automl]"

In [ ]:
!pip install mlflow

In [ ]:
# Step1: Data Ingestion
def data_ingestion():
  df = pd.read_csv(r'/content/loan_data.csv')
  df.shape
  return df

In [ ]:
# Checking Dataset Information
# df.info()

In [ ]:
'''
Note:
1. For Binary Classification: Sigmoid as activation function
2. For MultiClass Classification: Softmax as activation function
3. For Regression: Linear as activation function
4. For Binary Classification: BinaryCrossEntropy as loss function
5. For MultiClass Classification: CategoricalCrossEntropy as loss function
6. For Regression: MeanSquaredError as loss function
7. For Binary Classification: Accuracy as metric
8. For MultiClass Classification: Accuracy as metric
9. For Regression: MeanSquaredError as metric
10. Relu as activation function is used to avoid vanishing gradient problem and which impoved model performance in deep learning.

'''

'\nNote:\n1. For Binary Classification: Sigmoid as activation function\n2. For MultiClass Classification: Softmax as activation function\n3. For Regression: Linear as activation function\n4. For Binary Classification: BinaryCrossEntropy as loss function\n5. For MultiClass Classification: CategoricalCrossEntropy as loss function\n6. For Regression: MeanSquaredError as loss function\n7. For Binary Classification: Accuracy as metric\n8. For MultiClass Classification: Accuracy as metric\n9. For Regression: MeanSquaredError as metric\n10. Relu as activation function is used to avoid vanishing gradient problem and which impoved model performance in deep learning.\n\n'

In [ ]:
def preprocessing(df):

    # Remove duplicates
    df = df.drop_duplicates()

    # Numerical columns
    numerical_data = df.select_dtypes(exclude='object')

    # Winsorization
    for i in numerical_data.columns:
        if i != 'loan_status':
            df[i] = winsorize(
                df[i],
                limits=[0.05, 0.05]
            )

    # Split X and y
    X = df.drop(columns=['loan_status'])
    y = df['loan_status']

    # Train Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=1,
        stratify=y
    )

    return X_train, X_test, y_train, y_test

In [ ]:
# Install AutoGluon
!pip install -U "autogluon.tabular"

In [ ]:
from autogluon.tabular import TabularPredictor
from sklearn.metrics import accuracy_score, classification_report


def model_build(X_train, X_test, y_train, y_test):

    # Create Train DataFrame
    train_data = X_train.copy()     # Seen Data
    train_data['loan_status'] = y_train

    # Create Test DataFrame
    test_data = X_test.copy()       # Unseen Data
    test_data['loan_status'] = y_test

    # AutoGluon Model
    predictor = TabularPredictor(
        label='loan_status',
        problem_type='binary',
        eval_metric='accuracy'
    ).fit(
        train_data=train_data,
        presets='best_quality',
        time_limit=300
    )

    # Model Leaderboard
    print("\nModel Leaderboard")
    print(predictor.leaderboard(test_data, silent=True))

    # Prediction
    y_pred = predictor.predict(X_test)

    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print("\nAccuracy:", accuracy)

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return predictor

In [ ]:
def main():

    df = data_ingestion()

    X_train, X_test, y_train, y_test = preprocessing(df)

    predictor = model_build(
        X_train,
        X_test,
        y_train,
        y_test
    )


main()

No path specified. Models will be saved in: "AutogluonModels/ag-20260817_091612"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       10.77 GB / 12.67 GB (85.0%)
Disk Space Avail:   86.37 GB / 107.72 GB (80.2%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stac

[1000]	valid_set's binary_error: 0.0717143


	0.9321	 = Validation score   (accuracy)
	12.69s	 = Training   runtime
	0.51s	 = Validation runtime
Fitting model: RandomForestGini_BAG_L1 ... Training model for up to 17.37s of the 42.15s of remaining time.
	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=2, gpus=0, mem=0.1/10.6 GB
	0.9231	 = Validation score   (accuracy)
	6.39s	 = Training   runtime
	0.65s	 = Validation runtime
Fitting model: RandomForestEntr_BAG_L1 ... Training model for up to 9.99s of the 34.77s of remaining time.
	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=2, gpus=0, mem=0.1/10.5 GB
	0.9234	 = Validation score   (accuracy)
	6.5s	 = Training   runtime
	0.99s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ... Training model for up to 2.16s of the 26.93s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=0)
		`import catboost` failed. A quick tip is to install via `pip install autoglu


Model Leaderboard
                     model  score_test  score_val eval_metric  pred_time_test  \
0           XGBoost_BAG_L1    0.933481   0.933714    accuracy        0.838074   
1      WeightedEnsemble_L2    0.933481   0.933714    accuracy        0.839978   
2          LightGBM_BAG_L1    0.932963   0.931683    accuracy        1.866392   
3  RandomForestEntr_BAG_L1    0.927556   0.922984    accuracy        0.395958   
4  RandomForestGini_BAG_L1    0.926296   0.923746    accuracy        0.767911   
5        LightGBMXT_BAG_L1    0.925259   0.925333    accuracy        2.142259   
6    ExtraTreesGini_BAG_L1    0.923333   0.918825    accuracy        0.720713   
7    ExtraTreesEntr_BAG_L1    0.921481   0.919048    accuracy        0.497967   
8    NeuralNetTorch_BAG_L1    0.918222   0.918127    accuracy        0.383734   

   pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  \
0       0.244283   20.660653                 0.838074                0.244283   
1       